# WIB/FABRIC Live Demo Notebook

This is the front door for the artifact. It explains what the prototype does, what must already exist, how to run safe dry-runs, and how to move from a small smoke test to the full event 4245 replay.

WIB = Warm Interface Board.


## 1. Project Overview

### What this prototype does

- Replays a known event 4245 PCAP or captures packets already arriving on a receiver NIC.
- Preserves raw receiver-side PCAP chunks.
- Decodes each chunk through the Queue v2 path.
- Optionally validates decoded CSV rows against a reference CSV.
- Copies compact outputs back to the Jupyter/local machine.
- Shows cumulative notebook heatmap and ridgeline visualizations.

### What it does not do yet

- It does not create or modify FABRIC slices.
- It does not install VM packages or stage PCAP/reference files.
- It does not discover sender/receiver management hosts automatically for SSH.
- It does not fully verify external Jupyter unless you run it from a real external/local environment with real SSH details.

### Pipeline diagram

```text
pcap_replay: sender VM + staged PCAP + tcpreplay
                                         |
live_nic: packets already arriving on receiver NIC
                                         |
                                         v
receiver NIC -> raw PCAP chunks -> Queue v2 decode/export -> decoded CSV chunks
                                         |
                                         v
combined CSV -> validation summary -> notebook heatmap/ridgeline + timing CSV
```


## 2. Inputs And Assumptions

### Event 4245 replay config

The main reproducible config is `configs/event_4245_replay.yaml`. It uses `pcap_replay`, 640 packets, 64 packets per chunk, and therefore 10 chunks.

### Required files on sender and receiver VMs

| VM | Required items |
| --- | --- |
| Sender | `tcpreplay`, staged event PCAP at `pcap_replay.sender_pcap`, correct `network.sender_interface` |
| Receiver | `tcpdump`, Python 3, WIBViz source at `paths.wibviz_src`, mapping CSV at `paths.mapping_csv`, optional reference CSV at `validation.reference_csv`, correct `network.receiver_interface` |

### Required local files

Run from the artifact root, or start Jupyter from `notebooks/` and let the setup cell find the artifact root. Local Python needs `pyyaml`; visualization cells also use `pandas`, `matplotlib`, and IPython display helpers.

### Transport options

| Transport | Use when | Status |
| --- | --- | --- |
| `loom` | Running inside the LoomAI/FABRIC environment where LoomAI can reach the VMs | Most validated path |
| `ssh` | Running from standard FABRIC Jupyter, a laptop, or another local Jupyter environment | Dry-run verified only until tested with real SSH host/key details |


The next cell selects the package root, config file, transport, and safe execution flags. It does not contact VMs. For the first run, keep the defaults and run the dry-run cell next.


In [ ]:
from pathlib import Path
import json
import shlex
import subprocess
import sys

# Start Jupyter from the artifact root or notebooks/. Set this only if auto-detect fails.
package_root_override = ""  # example: "/Users/me/wib-fabric-live-demo"

if package_root_override:
    PACKAGE_ROOT = Path(package_root_override).expanduser().resolve()
else:
    current = Path.cwd().resolve()
    candidates = [current, *current.parents]
    PACKAGE_ROOT = next(
        (
            candidate
            for candidate in candidates
            if (candidate / "scripts/run_pipeline.py").is_file()
            and (candidate / "configs").is_dir()
        ),
        None,
    )
    if PACKAGE_ROOT is None:
        raise RuntimeError("Could not find the artifact root. Start Jupyter from the artifact root or notebooks/.")

# Main choices. Use "ssh" only after copying and editing an SSH config template.
source_mode = "pcap_replay"   # "pcap_replay" or "live_nic"
transport_backend = "loom"    # "loom" or "ssh"
demo = "event_4245"           # "event_4245", "custom_pcap", or "live_nic"

# For external SSH, copy a template first and set this, for example:
# config_path_override = "configs/event_4245_replay_ssh.yaml"
config_path_override = ""

# Safety defaults: dry-run is safe; smoke/full/backend runs require toggles below.
smoke_chunks = 2
run_smoke_test = False
run_full_replay = False
run_live_backend = False
enable_adaptive = False
validate_results = True
output_root_override = ""      # example: "/tmp/wib_fabric_live_demo"

config_map = {
    "loom": {
        "event_4245": PACKAGE_ROOT / "configs/event_4245_replay.yaml",
        "custom_pcap": PACKAGE_ROOT / "configs/custom_pcap_replay_template.yaml",
        "live_nic": PACKAGE_ROOT / "configs/live_nic_template.yaml",
    },
    "ssh": {
        "event_4245": PACKAGE_ROOT / "configs/event_4245_replay_ssh_template.yaml",
        "live_nic": PACKAGE_ROOT / "configs/live_nic_ssh_template.yaml",
    },
}

if transport_backend not in config_map:
    raise ValueError(f"unknown transport_backend: {transport_backend}")
selected_demo = "live_nic" if source_mode == "live_nic" else demo
if selected_demo not in config_map[transport_backend]:
    raise ValueError(f"{transport_backend} transport has no config for {selected_demo}")

if config_path_override:
    candidate = Path(config_path_override).expanduser()
    config_path = candidate if candidate.is_absolute() else PACKAGE_ROOT / candidate
else:
    config_path = config_map[transport_backend][selected_demo]

if not config_path.is_file():
    raise FileNotFoundError(f"Config does not exist: {config_path}")

print("Artifact root:", PACKAGE_ROOT)
print("Config:", config_path)
print("Source mode:", source_mode)
print("Transport:", transport_backend)
if transport_backend == "ssh" and config_path.name.endswith("_template.yaml"):
    print("SSH template selected. Dry-run can inspect it, but copy/edit it before smoke or full runs.")


This cell reads the selected YAML and prints the important local and VM path assumptions. It is a local-only sanity check.


In [ ]:
import yaml

with config_path.open(encoding="utf-8") as handle:
    selected_config = yaml.safe_load(handle)

summary = {
    "label": selected_config.get("label"),
    "event_id": selected_config.get("event_id"),
    "source_mode": selected_config.get("source_mode"),
    "transport": selected_config.get("transport", {}).get("type"),
    "slice": selected_config.get("slice"),
    "network": selected_config.get("network"),
    "local_output_root": selected_config.get("paths", {}).get("output_root"),
    "receiver_vm_paths": {
        "receiver_root": selected_config.get("paths", {}).get("receiver_root"),
        "wibviz_src": selected_config.get("paths", {}).get("wibviz_src"),
        "mapping_csv": selected_config.get("paths", {}).get("mapping_csv"),
        "reference_csv": selected_config.get("validation", {}).get("reference_csv"),
    },
    "sender_vm_pcap": selected_config.get("pcap_replay", {}).get("sender_pcap"),
}
print(json.dumps(summary, indent=2, sort_keys=True))


## 3. Quick Start Inside FABRIC/LoomAI

Run these from the artifact root when using the validated LoomAI path.

Dry-run only:

```bash
python3 scripts/run_pipeline.py --config configs/event_4245_replay.yaml --dry-run
```

Two-chunk smoke test after dry-run passes:

```bash
python3 scripts/run_pipeline.py --config configs/event_4245_replay.yaml --max-live-chunks 2
```

Full 10-chunk event 4245 run after the smoke test passes:

```bash
python3 scripts/run_pipeline.py --config configs/event_4245_replay.yaml
```

The helper below runs the same CLI from this notebook. Dry-run is local-only and safe.


In [ ]:
def pipeline_command(*, dry_run=False, extra_args=None):
    cmd = [
        sys.executable,
        str(PACKAGE_ROOT / "scripts/run_pipeline.py"),
        "--config",
        str(config_path),
        "--source-mode",
        source_mode,
    ]
    if dry_run:
        cmd.append("--dry-run")
    if output_root_override:
        cmd.extend(["--output-root", str(Path(output_root_override).expanduser())])
    if enable_adaptive:
        cmd.append("--adaptive")
    if not validate_results:
        cmd.append("--no-validate")
    if extra_args:
        cmd.extend(extra_args)
    return cmd


def run_pipeline(*, dry_run=False, extra_args=None):
    cmd = pipeline_command(dry_run=dry_run, extra_args=extra_args)
    print("Command:", " ".join(shlex.quote(str(part)) for part in cmd))
    result = subprocess.run(cmd, cwd=str(PACKAGE_ROOT), text=True, capture_output=True)
    if result.stderr.strip():
        print("stderr:")
        print(result.stderr)
    if result.returncode != 0:
        print("stdout:")
        print(result.stdout)
        raise RuntimeError(f"runner failed with exit code {result.returncode}")
    payload = json.loads(result.stdout)
    compact = {
        "dry_run": payload.get("dry_run"),
        "run_id_or_dir": payload.get("run_id") or payload.get("local_paths", {}).get("run_dir"),
        "status": payload.get("status"),
        "pass": payload.get("pass"),
        "manifest": payload.get("manifest") or payload.get("local_paths", {}).get("manifest_json"),
        "summary": payload.get("summary") or payload.get("local_paths", {}).get("summary_json"),
    }
    print(json.dumps(compact, indent=2, sort_keys=True))
    return payload


Run the dry-run first. It parses the config, writes a local manifest/dry-run JSON under `paths.output_root`, and does not SSH to VMs or run packet tools.


In [ ]:
dry_payload = run_pipeline(dry_run=True)
payload = dry_payload
print("Dry-run JSON:", dry_payload["local_paths"]["dry_run_json"])
print("Manifest:", dry_payload["local_paths"]["manifest_json"])


This optional smoke-test cell is the first VM-contacting run. It is disabled by default. To use it, set `run_smoke_test = True` in the setup cell after the dry-run and VM path checks pass.


In [ ]:
smoke_args = ["--max-live-chunks", str(smoke_chunks)]
smoke_cmd = pipeline_command(extra_args=smoke_args)
print("Copy/paste smoke-test command:")
print(" ".join(shlex.quote(str(part)) for part in smoke_cmd))

if run_smoke_test:
    payload = run_pipeline(extra_args=smoke_args)
else:
    print("Smoke test not run. Set run_smoke_test = True only after dry-run and VM checks pass.")


## 4. External Jupyter Setup

Use this path from a laptop, desktop, or standard FABRIC Jupyter environment. The artifact's SSH workflow is dry-run verified only until you run it with real SSH details.

### Generate or choose an SSH key

```bash
ssh-keygen -t ed25519 -f ~/.ssh/fabric_wibviz -C "fabric-wibviz"
cat ~/.ssh/fabric_wibviz.pub
```

Register or use the public key through the FABRIC/project access mechanism. Keep the private key local.

### Find sender and receiver management hosts

Use VM management IPs, DNS names, or SSH config aliases. Do not use detector/data-plane IPs unless your environment explicitly routes them for management.

```bash
SENDER_HOST=<sender-management-host-or-ip>
RECEIVER_HOST=<receiver-management-host-or-ip>
ssh -i ~/.ssh/fabric_wibviz ubuntu@$SENDER_HOST hostname
ssh -i ~/.ssh/fabric_wibviz ubuntu@$RECEIVER_HOST hostname
```

If needed, use ProxyJump or a FABRIC SSH config:

```bash
ssh -i ~/.ssh/fabric_wibviz -J <bastion-user>@<bastion-host> ubuntu@$SENDER_HOST hostname
```

### Fill in the SSH config template

```bash
cp configs/event_4245_replay_ssh_template.yaml configs/event_4245_replay_ssh.yaml
```

Edit the copied YAML and set `transport.username`, `transport.sender_host`, `transport.receiver_host`, `transport.ssh_key`, optional `transport.ssh_options`, data-plane interfaces, `pcap_replay.sender_pcap`, `paths.wibviz_src`, `paths.mapping_csv`, and `validation.reference_csv` if validation is enabled.

External dry-run:

```bash
python3 scripts/run_pipeline.py --config configs/event_4245_replay_ssh.yaml --dry-run
```

External smoke test after raw SSH and dry-run pass:

```bash
python3 scripts/run_pipeline.py --config configs/event_4245_replay_ssh.yaml --max-live-chunks 2
```


## 5. Running The Live Notebook Demo

The live notebook demo watches decoded chunk CSVs while the backend writes them. Start the viewer first, then start the backend. For the default event 4245 config, expected progress is 10 chunks and 640 packets.

Chunks are packet batches, not tick/time slices. Each packet batch may cover the full tick range while covering only part of the wire range, so plots can fill by horizontal wire bands rather than left-to-right in time.


This cell builds a unique live run ID and backend command. It does not start the viewer or backend yet.


In [ ]:
from datetime import datetime, timezone

live_config = yaml.safe_load(config_path.read_text(encoding="utf-8"))
if live_config.get("source_mode") != "pcap_replay":
    print("Selected config is not pcap_replay; event 4245 expectations may not apply.")

live_demo_mode = "full"  # "full" = 10 chunks / 640 packets; "smoke" = 2 chunks / 128 packets
live_smoke_chunks = 2
live_full_event_chunks = 10
live_chunk_packets = int(live_config.get("decode", {}).get("chunking", {}).get("chunk_packets") or 64)
live_total_packets = int(live_config.get("pcap_replay", {}).get("packet_limit") or live_full_event_chunks * live_chunk_packets)
live_max_chunks = live_full_event_chunks if live_demo_mode == "full" else live_smoke_chunks
live_expected_packets = min(live_total_packets, live_max_chunks * live_chunk_packets)
live_poll_seconds = 2.0

live_tick_range = (0, 2047)
live_wire_range = (0, 199)
live_color_limits = (-20, 50)
live_waveform_mode = "full_range"
live_waveform_wire_range = live_wire_range
live_waveform_sample_count = None
live_waveform_adc_scale = 0.01

live_run_id = f"notebook-live-4245-{live_demo_mode}-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
live_output_root = Path(output_root_override).expanduser() if output_root_override else Path(live_config.get("paths", {}).get("output_root", "/tmp/wib_fabric_live_demo")).expanduser()
live_run_dir = live_output_root / "runs" / live_run_id
live_decoded_chunk_dir = live_run_dir / "artifacts" / "decoded_chunks"

live_backend_cmd = pipeline_command(extra_args=["--run-id", live_run_id, "--max-live-chunks", str(live_max_chunks)])
print("Live mode:", live_demo_mode)
print("Expected chunks:", live_max_chunks)
print("Expected packets:", live_expected_packets)
print("Live run directory:", live_run_dir)
print("Backend command:")
print(" ".join(shlex.quote(str(part)) for part in live_backend_cmd))


Start the notebook live viewer with this cell. It watches the local run directory and updates when decoded chunk CSVs appear.


In [ ]:
import importlib.util
import inspect

viewer_module_path = (PACKAGE_ROOT / "scripts" / "live_chunk_viewer.py").resolve()
if not viewer_module_path.is_file():
    raise FileNotFoundError(f"Missing live viewer helper: {viewer_module_path}")

spec = importlib.util.spec_from_file_location("live_chunk_viewer", viewer_module_path)
live_chunk_viewer = importlib.util.module_from_spec(spec)
sys.modules["live_chunk_viewer"] = live_chunk_viewer
spec.loader.exec_module(live_chunk_viewer)
start_notebook_live_viewer = live_chunk_viewer.start_notebook_live_viewer
print("Viewer helper:", live_chunk_viewer.__file__)
print("Signature:", inspect.signature(start_notebook_live_viewer))

live_viewer = start_notebook_live_viewer(
    live_run_dir,
    poll_seconds=live_poll_seconds,
    label=f"event 4245 {live_run_id}",
    stop_when_complete=True,
    expected_total_chunks=live_max_chunks,
    chunk_packets=live_chunk_packets,
    total_packets=live_expected_packets,
    tick_range=live_tick_range,
    wire_range=live_wire_range,
    color_limits=live_color_limits,
    waveform_mode=live_waveform_mode,
    waveform_wire_range=live_waveform_wire_range,
    waveform_window_size=live_waveform_sample_count,
    waveform_adc_scale=live_waveform_adc_scale,
)
print("Live viewer started. Now start the backend from a terminal or by enabling run_live_backend.")


Start the backend only after the viewer is running and the dry-run/smoke-test checks are acceptable. This cell is disabled by default; set `run_live_backend = True` in the setup cell or paste the printed command into a terminal.


In [ ]:
print("Backend command:")
print(" ".join(shlex.quote(str(part)) for part in live_backend_cmd))

if transport_backend == "ssh" and config_path.name.endswith("_template.yaml"):
    raise RuntimeError("Copy and edit the SSH template before starting an SSH backend run.")

if run_live_backend:
    live_reports_dir = live_run_dir / "reports"
    live_reports_dir.mkdir(parents=True, exist_ok=True)
    live_backend_stdout_path = live_reports_dir / "notebook_backend_stdout.log"
    live_backend_stderr_path = live_reports_dir / "notebook_backend_stderr.log"
    live_backend_stdout_handle = live_backend_stdout_path.open("w", encoding="utf-8", buffering=1)
    live_backend_stderr_handle = live_backend_stderr_path.open("w", encoding="utf-8", buffering=1)
    live_backend_process = subprocess.Popen(
        live_backend_cmd,
        cwd=str(PACKAGE_ROOT),
        text=True,
        stdout=live_backend_stdout_handle,
        stderr=live_backend_stderr_handle,
    )
    print("Started backend PID:", live_backend_process.pid)
    print("stdout log:", live_backend_stdout_path)
    print("stderr log:", live_backend_stderr_path)
else:
    print("Backend not started. Set run_live_backend = True or paste the command into a terminal when ready.")


Use this cell after the backend finishes. It reads the manifest, summary, decoded chunks, and live timing CSV if they exist.


In [ ]:
process = globals().get("live_backend_process")
if process is not None:
    rc = process.wait()
    for handle_name in ("live_backend_stdout_handle", "live_backend_stderr_handle"):
        handle = globals().get(handle_name)
        if handle and not handle.closed:
            handle.close()
    print("Backend exit code:", rc)
else:
    rc = None
    print("No notebook backend process found; reading current run directory status only.")

manifest_path = live_run_dir / "reports" / "manifest.json"
summary_path = live_run_dir / "reports" / "summary.json"
timing_path = live_run_dir / "reports" / "live_visualization_timing.csv"
if not manifest_path.exists():
    print("Manifest not found yet:", manifest_path)
else:
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    summary = json.loads(summary_path.read_text(encoding="utf-8")) if summary_path.exists() else {}
    decoded_chunks = sorted(str(path) for path in live_decoded_chunk_dir.glob("chunk_*.csv"))
    print(json.dumps({
        "run_id": manifest.get("run_id"),
        "status": manifest.get("status"),
        "phase": manifest.get("phase"),
        "pass": manifest.get("pass"),
        "validation_result": manifest.get("validation_result"),
        "queue_metrics": manifest.get("queue_metrics"),
        "decoded_chunk_count": len(decoded_chunks),
        "expected_chunks": live_max_chunks,
        "expected_packets": live_expected_packets,
        "timing_csv": str(timing_path) if timing_path.exists() else None,
        "summary": summary,
    }, indent=2, sort_keys=True))
    if rc not in (None, 0):
        raise RuntimeError(f"Backend failed with exit code {rc}.")


## 6. Outputs

Local outputs default to `/tmp/wib_fabric_live_demo/runs/<run_id>/` unless `paths.output_root` or `--output-root` changes it.

| Output | Meaning |
| --- | --- |
| `reports/manifest.json` | status, phase, pass/fail, paths, chunks, queue metrics, validation result |
| `reports/live_visualization_timing.csv` | notebook live-viewer timing and lag records |
| `raw_chunks/` on receiver output path | preserved raw PCAP chunks |
| `artifacts/decoded_chunks/chunk_*.csv` | local decoded CSV chunks |
| `artifacts/combined.csv` | final combined decoded CSV |
| `visualizations/heatmap.png` | final heatmap |
| `visualizations/stacked_waveform.png` | final ridgeline-style figure |
| `visualizations/live_chunks/` | cumulative live-viewer heatmap/ridgeline images |

Remote receiver paths are under `<receiver_root>/wibviz_runs/wib_fabric_live_demo/<run_id>/` and `<receiver_root>/wibviz_outputs/wib_fabric_live_demo/<run_id>/`.


This optional full-run cell is disabled by default. Use it only after dry-run and the two-chunk smoke test pass. The default event 4245 config runs 10 chunks and 640 packets.


In [ ]:
full_cmd = pipeline_command()
print("Copy/paste full-run command:")
print(" ".join(shlex.quote(str(part)) for part in full_cmd))

if run_full_replay:
    if source_mode != "pcap_replay":
        raise RuntimeError("This full-run cell is for pcap_replay. For live_nic, use bounded --max-live-chunks or intentional --follow.")
    payload = run_pipeline()
else:
    print("Full replay not run. Set run_full_replay = True only after dry-run and smoke test pass.")


## 7. Interpreting Results

Key fields to inspect:

- `pass`: overall runner pass/fail when a real run completes.
- `validation_result`: generated CSV vs reference CSV status, including missing/extra rows.
- `raw_packet_drops`: packet drops reported from capture logs when available.
- `queue_metrics`: queue depth and decode/copy timing indicators.
- `capture-to-visualization lag`: time from receiver capture progress to notebook update.
- `decode duration`: time spent converting raw chunk PCAP to decoded CSV.
- `receiver-to-local copy time`: transfer duration for chunk outputs.
- `visualization lag`: time for local viewer rendering after chunks appear.

The next cell summarizes the latest `payload` or dry-run manifest available in the notebook.


In [ ]:
active_payload = globals().get("payload", globals().get("dry_payload"))
if active_payload is None:
    raise RuntimeError("Run the dry-run cell first.")

if active_payload.get("dry_run"):
    manifest_path = Path(active_payload["local_paths"]["manifest_json"])
else:
    manifest_path = Path(active_payload.get("manifest") or active_payload.get("local_paths", {}).get("manifest_json"))

manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
metrics = {
    "dry_run": bool(active_payload.get("dry_run")),
    "manifest": str(manifest_path),
    "run_id": manifest.get("run_id"),
    "status": manifest.get("status"),
    "phase": manifest.get("phase"),
    "pass": manifest.get("pass"),
    "validation_result": manifest.get("validation_result"),
    "queue_metrics": manifest.get("queue_metrics"),
    "raw_chunk_count": len(manifest.get("raw_chunks", [])),
    "decoded_chunk_count": len(manifest.get("chunks", [])),
    "combined_csv": manifest.get("paths", {}).get("local_combined_csv"),
}
print(json.dumps(metrics, indent=2, sort_keys=True))


Display final heatmap and ridgeline images if a real smoke/full run has produced them. Dry-run alone will not create these images.


In [ ]:
from IPython.display import Image, display

active_payload = globals().get("payload", globals().get("dry_payload"))
if active_payload is None:
    raise RuntimeError("Run the dry-run cell first.")

if active_payload.get("dry_run"):
    viz_dir = Path(active_payload["local_paths"]["visualization_dir"])
else:
    viz_dir = Path(active_payload.get("local_run_dir") or active_payload.get("local_paths", {}).get("run_dir")) / "visualizations"

heatmap = viz_dir / "heatmap.png"
ridgeline = viz_dir / "stacked_waveform.png"
print("Visualization directory:", viz_dir)

if heatmap.exists() and ridgeline.exists():
    print("Heatmap:", heatmap)
    display(Image(filename=str(heatmap)))
    print("Ridgeline:", ridgeline)
    display(Image(filename=str(ridgeline)))
else:
    print("No final heatmap/ridgeline images found yet. Run a real smoke/full pipeline first.")
    print("Expected heatmap:", heatmap)
    print("Expected ridgeline:", ridgeline)


## 8. Troubleshooting

| Symptom | First check |
| --- | --- |
| Wrong working directory | Start Jupyter from artifact root or `notebooks/`, or set `package_root_override`. |
| Invalid notebook on GitHub | Re-run notebook validation with `nbformat`; keep outputs stripped before committing. |
| SSH failure | Confirm `transport.username`, `transport.ssh_key`, public-key registration, hostnames, and `ssh_options` for ProxyJump or SSH config. |
| Missing VM files | Check `pcap_replay.sender_pcap`, `paths.wibviz_src`, `paths.mapping_csv`, and `validation.reference_csv` on the VMs. |
| Wrong interface | Run `ip -brief addr` on each VM and update `network.sender_interface` or `network.receiver_interface`. |
| No plots appearing | Confirm a real run produced decoded chunk CSVs under `artifacts/decoded_chunks/`; dry-run does not create plots. |
| Plots fill by wire bands | This is expected for packet batches; chunks are not tick/time slices. |
| Slow visualization | Check `reports/live_visualization_timing.csv` for decode, copy, render, and end-to-end lag. |


## 9. Limitations And Next Steps

- External Jupyter full testing still requires real sender/receiver management hosts, a working SSH key, and staged VM-side files.
- Real live NIC input requires a prepared packet source feeding the receiver data-plane NIC.
- File-based decoded CSV handoff is simple and auditable but adds latency.
- Adaptive chunking is experimental/future-facing and disabled by default.
- Slice provisioning, package installation, host discovery, and input-file staging remain manual.

Supporting docs:

- [README](../README.md)
- [External Jupyter setup](../docs/external_jupyter_setup.md)
- [Standard FABRIC transport](../docs/standard_fabric_transport.md)
- [Troubleshooting](../docs/troubleshooting.md)
